# Sampling Project Template

Copy this notebook as the starting point for a new sampling project. It is
dataset-agnostic: fill in the **CONFIG** cell, then run the rest as-is. The
sampling functions are identical to those in the Cheat Sheet, packaged for
reuse.


## 0. Config — edit this cell for your project

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# ---- EDIT ME -------------------------------------------------------------
DATA_PATH   = "your_data.csv"     # path to your population / sampling frame
TARGET_COL  = "value"             # numeric column you want to estimate
STRATA_COL  = None                # categorical column to stratify on, or None
CLUSTER_COL = None                # column that identifies natural clusters, or None
SAMPLE_N    = 100                 # desired sample size
RANDOM_SEED = 2020
# ---------------------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)


## 1. Load data

In [ ]:
# df = pd.read_csv(DATA_PATH)
# df.head()

# For a first dry run without real data, uncomment to generate a toy frame:
# df = pd.DataFrame({
#     TARGET_COL: rng.normal(50, 10, size=1000),
#     "strata_demo": rng.choice(["A", "B", "C"], size=1000),
#     "cluster_demo": np.arange(1000) // 25,
# })


## 2. Sampling function library
(Identical to the cheat sheet — copy/paste-safe across projects.)

In [ ]:
def simple_random_sample(df, n, rng):
    idx = rng.choice(df.index, size=n, replace=False)
    return df.loc[idx]

def systematic_sample(df, n, rng):
    N = len(df)
    k = max(N // n, 1)
    start = rng.integers(0, k)
    return df.iloc[np.arange(start, N, k)[:n]]

def stratified_sample(df, strata_col, frac=None, n_per_stratum=None, rng=None):
    """Provide either `frac` (proportional allocation) or
    `n_per_stratum` (dict of {level: n}, fixed allocation)."""
    parts = []
    for level, group in df.groupby(strata_col):
        n_g = int(round(frac * len(group))) if frac is not None else n_per_stratum[level]
        n_g = min(n_g, len(group))
        idx = rng.choice(group.index, size=n_g, replace=False)
        parts.append(df.loc[idx])
    return pd.concat(parts)

def cluster_sample(df, cluster_col, n_clusters, rng, stage2_frac=None):
    clusters = df[cluster_col].unique()
    n_clusters = min(n_clusters, len(clusters))
    chosen = rng.choice(clusters, size=n_clusters, replace=False)
    sample = df[df[cluster_col].isin(chosen)]
    if stage2_frac is not None:
        sample = stratified_sample(sample, cluster_col, frac=stage2_frac, rng=rng)
    return sample

def convenience_sample(df, n, filter_col=None, filter_value=None):
    pool = df if filter_col is None else df[df[filter_col] == filter_value]
    return pool.head(n)

def quota_sample(df, strata_col, quotas):
    parts = [df[df[strata_col] == lvl].head(n_g) for lvl, n_g in quotas.items()]
    return pd.concat(parts)

def mean_ci(sample_values, alpha=0.05):
    x_bar = sample_values.mean()
    s = sample_values.std(ddof=1)
    n_obs = len(sample_values)
    t_crit = stats.t.ppf(1 - alpha / 2, df=n_obs - 1)
    margin = t_crit * s / np.sqrt(n_obs)
    return x_bar, (x_bar - margin, x_bar + margin)


## 3. Choose and draw your sample

Uncomment the block that matches your design. Only one is usually needed.


In [ ]:
# -- Simple random --------------------------------------------------------
# sample = simple_random_sample(df, n=SAMPLE_N, rng=rng)

# -- Systematic -------------------------------------------------------------
# sample = systematic_sample(df, n=SAMPLE_N, rng=rng)

# -- Stratified (proportional) ---------------------------------------------
# sample = stratified_sample(df, strata_col=STRATA_COL,
#                             frac=SAMPLE_N / len(df), rng=rng)

# -- Stratified (fixed per-stratum quota, still random within stratum) -----
# sample = stratified_sample(df, strata_col=STRATA_COL,
#                             n_per_stratum={"A": 30, "B": 30, "C": 30}, rng=rng)

# -- Cluster ----------------------------------------------------------------
# sample = cluster_sample(df, cluster_col=CLUSTER_COL, n_clusters=10, rng=rng)

# sample.head()


## 4. Sanity checks

In [ ]:
# assert len(sample) > 0, "Sample is empty -- check your config"
# print(f"Sample size: {len(sample)}")
# if STRATA_COL:
#     print(sample.groupby(STRATA_COL).size())
# sample[TARGET_COL].describe()


## 5. Estimate + confidence interval

In [ ]:
# x_bar, (ci_low, ci_high) = mean_ci(sample[TARGET_COL])
# print(f"Estimated mean: {x_bar:.3f}")
# print(f"95% CI: ({ci_low:.3f}, {ci_high:.3f})")


## 6. (Optional) Monte Carlo self-check

If you have — or can simulate — a known ground truth, use this block to
sanity-check that your chosen method is behaving as expected before
trusting it on the real project.


In [ ]:
# TRUE_MEAN = df[TARGET_COL].mean()  # only meaningful if `df` IS the full population
# means = []
# for _ in range(500):
#     s = simple_random_sample(df, n=SAMPLE_N, rng=rng)  # swap in your method
#     means.append(s[TARGET_COL].mean())
# means = np.array(means)
# print(f"Bias: {means.mean() - TRUE_MEAN:.4f}")
# print(f"Std dev of estimate: {means.std(ddof=1):.4f}")


## Notes for next time

- Record here: what was `STRATA_COL` / `CLUSTER_COL` for this project, and
  why that choice made sense.
- Record the achieved sample size vs. target, and any deviations.
- Record whether this was a probability sample (formal inference valid) or
  non-probability (exploratory only) and why.
